<a href="https://colab.research.google.com/github/Tucannsam/Recommendation_System/blob/main/notebooks/03_low_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PART A
import os
import subprocess
import sys

USER = Tucannsam
REPO = Recommendation_System

if os.path.exists(REPO):
subprocess.run([git, -C, REPO, pull], check=True)
else:
subprocess.run(
[git, clone, fhttps://github.com/{USER}/{REPO}.git, REPO],
check=True)
sys.path.insert(0, f{REPO}/src)
from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline

This code gets my project from GitHub and updates it if I already have it downloaded. It also imports the functions I need for the rest of the notebook.

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists(ml-100k):
urllib.request.urlretrieve(
https://files.grouplens.org/datasets/movielens/ml-100k.zip,
ml-100k.zip)
zipfile.ZipFile(ml-100k.zip).extractall()

ratings = pd.read_csv(
ml-100k/u.data, sep=\\t,
names=[user, item, rating, time])

train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)
print(fit:, len(fit), validation:, len(val), test:, len(test))

This code downloads the movie ratings if they are not already there. I split the ratings into fit, validation, and test sets so I can compare the models before checking the final results.

In [ ]:
#PART B
R_fit = to_matrix(fit)
known_fit = ~np.isnan(R_fit)
val_truth = val[rating].to_numpy()
val_u = val[user].to_numpy() - 1
val_i = val[item].to_numpy() - 1

def score_at(P, rows, cols, truth):
RMSE for selected entries of a prediction matrix.
pred = np.clip(P[rows, cols], 1, 5)
return rmse(pred, truth)

This code puts the ratings into a matrix and keeps track of which ones are known. I also make a function to calculate RMSE and keep the predicted ratings between 1 and 5.

In [ ]:
#PART C
def baseline_matrix(R):
  Full matrix: overall mean plus user and movie effects.
  mu, b_user, b_item = bias_baseline(R)
  return mu + b_user[:, None] + b_item[None, :]

B_fit = baseline_matrix(R_fit)
baseline_val_rmse = score_at(B_fit, val_u, val_i, val_truth)
print(Baseline validation RMSE:, round(baseline_val_rmse, 4))

This code makes the baseline predictions using the overall average and the user and movie biases. I check the validation error so I can compare it to the other models later.

In [ ]:
#PART D
def rank_k(U, s, Vt, k):
  Build a rank-k matrix from an already computed SVD.
  return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

This function uses the first k parts of the SVD to rebuild the matrix. A smaller k keeps fewer patterns, while a larger k keeps more.

In [ ]:
#Part E
mu_fit = np.nanmean(R_fit)
n_item = known_fit.sum(axis=0)
item_mean = np.full(R_fit.shape[1], mu_fit)
has_rating = n_item > 0
item_mean[has_rating] = (
  np.nansum(R_fit, axis=0)[has_rating] / n_item[has_rating])

F = np.where(known_fit, R_fit, item_mean[None, :])
U1, s1, Vt1 = np.linalg.svd(F, full_matrices=False)

This code fills in missing ratings with the average for that movie, then runs SVD. If the movie has no ratings, I use the overall average.

In [ ]:
#PART F
X_fit = np.where(known_fit, R_fit - B_fit, 0.0)
U2, s2, Vt2 = np.linalg.svd(X_fit, full_matrices=False)

This code finds the difference between the actual ratings and the baseline predictions. I use zero for the missing values and then run SVD on those differences.

In [ ]:
#PART G
ks = [1, 2, 3, 5, 10, 15, 20, 30, 50, 100]
val_err1, val_err2 = [], []

for k in ks:
  P1 = rank_k(U1, s1, Vt1, k)
  P2 = rank_k(U2, s2, Vt2, k) + B_fit
  val_err1.append(score_at(P1, val_u, val_i, val_truth))
  val_err2.append(score_at(P2, val_u, val_i, val_truth))

best_k1 = ks[int(np.argmin(val_err1))]
best_k2 = ks[int(np.argmin(val_err2))]
print(Best Model 1 k:, best_k1, validation RMSE:, min(val_err1))
print(Best Model 2 k:, best_k2, validation RMSE:, min(val_err2))

This code tries different k values for both models. I compare their validation errors and pick the k that gives the lowest error for each one.

In [ ]:
plt.semilogx(ks, val_err1, o-, label=fill with movie averages)
plt.semilogx(ks, val_err2, s-, label=SVD of baseline deviations)
plt.axhline(baseline_val_rmse, color=gray, linestyle=--,label=baseline)
plt.xlabel(rank k (log scale))
plt.ylabel(validation RMSE (stars))
plt.title(Choose rank using validation data)
plt.grid()
plt.legend()
plt.savefig(validation_rmse_vs_k.png, dpi=150)
plt.show()

This graph shows how the validation error changes with k for both models. I include the baseline error to see how they compare.

In [ ]:
#PART H
def iterative_svd_predict(R, k, n_iter=10):
  Predict ratings by repeatedly imputing missing deviations.
  known = ~np.isnan(R)

  B = baseline_matrix(R)
  X = np.where(known, R - B, 0.0)

  for _ in range(n_iter):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    L = rank_k(U, s, Vt, k)
    X = np.where(known, R - B, L)

return L + B

This function runs SVD multiple times to update the missing ratings. It keeps the known ratings the same and adds the baseline back at the end.

In [ ]:
ks_iter = [2, 5, 10, 20]
val_err3 = []

for k in ks_iter:
  P3 = iterative_svd_predict(R_fit, k, n_iter=10)
  val_err3.append(score_at(P3, val_u, val_i, val_truth))

best_k3 = ks_iter[int(np.argmin(val_err3))]
print(Best iterative rank:, best_k3)
print(Best validation RMSE:, min(val_err3))

This code tries a few k values for the iterative model, with 10 rounds for each one. I pick the k with the lowest validation error.

In [ ]:
#PART I
R_train = to_matrix(train)
test_truth = test[rating].to_numpy()
test_u = test[user].to_numpy() - 1
test_i = test[item].to_numpy() - 1

B_train = baseline_matrix(R_train)
test_baseline = score_at(B_train, test_u, test_i, test_truth)

if min(val_err3) < min(val_err2) and min(val_err3) < min(val_err1):
  test_prediction = iterative_svd_predict(
    R_train, best_k3, n_iter=10)
  chosen_model = iterative SVD
elif min(val_err2) < min(val_err1):
  known_train = ~np.isnan(R_train)
  X_train = np.where(known_train, R_train - B_train, 0.0)
  U, s, Vt = np.linalg.svd(X_train, full_matrices=False)
  test_prediction = rank_k(U, s, Vt, best_k2) + B_train
  chosen_model = baseline plus low-rank deviations
else:
  known_train = ~np.isnan(R_train)
  mu_train = np.nanmean(R_train)
  n_item_train = known_train.sum(axis=0)
  means_train = np.full(R_train.shape[1], mu_train)
  has_train = n_item_train > 0
  means_train[has_train] = (
    np.nansum(R_train, axis=0)[has_train] / n_item_train[has_train])
  F_train = np.where(known_train, R_train, means_train[None, :])
  U, s, Vt = np.linalg.svd(F_train, full_matrices=False)
  test_prediction = rank_k(U, s, Vt, best_k1)
  chosen_model = SVD of movie-average-filled matrix

test_rmse = score_at(test_prediction, test_u, test_i, test_truth)
print(Chosen model:, chosen_model)
print(Baseline test RMSE:, round(test_baseline, 4))
print(Chosen model test RMSE:, round(test_rmse, 4))

This code picks the SVD model that did best on the validation set and rebuilds it with the full training data. I then check its test error and compare it to the baseline.

In [ ]:
#PART J
item_info = pd.read_csv(
  ml-100k/u.item, sep=|, header=None, usecols=[0, 1],
  encoding=latin-1)
item_info.columns = [item, title]

user_id = 1
user_row = user_id - 1
unrated = np.where(np.isnan(R_train[user_row]))[0]
predicted_scores = np.clip(test_prediction[user_row], 1, 5)
top = unrated[np.argsort(predicted_scores[unrated])[-5:][::-1]]

for col in top:
  title = item_info.loc[item_info[item] == col + 1, title].iloc[0]
  print(title, predicted rating:, round(predicted_scores[col], 2))

This code finds five movies to recommend to user 1 based on the predicted ratings. It only picks movies the user has not rated in the training data and prints their titles and scores.